# Notebook 2 — Architecture, Training & Evaluation
## AlphaGenome × scRNA-seq Finetuning Pipeline

**Requires**: Notebook 1 completed and outputs saved to Google Drive.

### Architecture overview
```
DNA sequence (1 Mb)
      │
      ▼
  AlphaGenome API          ← fixed backbone (API prototype)
      │                      (swap to local LoRA weights once released)
      │  seq_emb: (1 Mb × 667 RNA-seq tracks)
      ▼
CellSpecificDecoder
  HypernetworkMLP          ← trained: 14 → 64 → 128 → 256 → (667+1)
  cell_emb: (n_cells × 14)   cell latent from scVI
      │
      ▼
  predicted coverage       ← (n_cells × exon_positions), squashed scale
```

### Evaluation metrics (matching scooby paper)
1. **Across-gene Pearson R** (log2 scale) per cell type
2. **Between-cell-type deviation R** — after removing gene and cell means

---
### ⚠️ Critical notes
- `ALPHAGENOME_API_KEY` must be set as a Colab Secret.
- Sequence embeddings are cached to Drive to avoid repeated API calls.
- Cell 14 (`load_checkpoint`) is **self-contained** — run it alone after reconnection.
- `n_seq_features=667` for API; change to `1920` once local weights are available.

## 0 — Mount Drive & Install Dependencies

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

DRIVE_ROOT = '/content/drive/MyDrive/alphagenome_finetune'
DATA_DIR   = f'{DRIVE_ROOT}/data/neurips'
CACHE_DIR  = f'{DRIVE_ROOT}/data/embedding_cache'
SAVE_DIR   = f'{DRIVE_ROOT}/checkpoints'

for d in [DATA_DIR, CACHE_DIR, SAVE_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')

In [ ]:
!pip install -q alphagenome scvi-tools scanpy anndata pyranges
!pip install -q git+https://github.com/lauradmartens/SnapATAC2.git@scooby

print('Packages installed.')

## 1 — Imports & Device Setup

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import anndata as ad
import json, os
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr

# AlphaGenome
from alphagenome.data import genome, gene_annotation
from alphagenome.models import dna_client
from google.colab import userdata

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 2 — Load Preprocessed Data (from Notebook 1)

In [ ]:
MANIFEST_PATH = f'{DATA_DIR}/preprocessing_manifest.json'

with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

print(json.dumps(manifest, indent=2))

In [ ]:
import snapatac2 as snap

# Cell embeddings: (n_cells, 14)
embedding_df    = pd.read_parquet(manifest['files']['cell_embeddings'])
cell_embeddings = torch.tensor(
    embedding_df[[f'z_{i}' for i in range(manifest['embedding_dim'])]].values,
    dtype=torch.float32
)
cell_barcodes   = embedding_df.index.tolist()
cell_types      = embedding_df['cell_type'].tolist()
print(f'Cell embeddings : {cell_embeddings.shape}')
print(f'Unique cell types: {len(set(cell_types))}')

# RNA coverage AnnData
rna_data = snap.read(manifest['files']['rna_coverage'], backed='r')
print(f'RNA coverage    : {rna_data.n_obs} cells × {rna_data.n_vars} positions')

# Gene intervals
gene_df     = pd.read_parquet(manifest['files']['gene_intervals'])
exon_df     = pd.read_parquet(manifest['files']['exon_coordinates'])
train_genes = gene_df[gene_df['split'] == 'train'].reset_index(drop=True)
val_genes   = gene_df[gene_df['split'] == 'val'].reset_index(drop=True)
test_genes  = gene_df[gene_df['split'] == 'test'].reset_index(drop=True)
print(f'Genes  train/val/test: {len(train_genes)}/{len(val_genes)}/{len(test_genes)}')
print(f'Exon records         : {len(exon_df)}')

## 3 — AlphaGenome API Client

In [ ]:
# Store your key at: Colab → Tools → Secrets → ALPHAGENOME_API_KEY
API_KEY   = userdata.get('ALPHAGENOME_API_KEY')
dna_model = dna_client.create(API_KEY)
print('AlphaGenome client ready.')
print(f'Sequence length: {dna_client.SEQUENCE_LENGTH_1MB:,} bp')

## 4 — Coverage Extraction Utilities (1 bp resolution)

In [ ]:
import snapatac2 as snap


def extract_coverage_1bp(
    rna_data,
    cell_indices: list,
    chromosome: str,
    start: int,
    end: int,
    clip_soft: float = 5.0,
) -> torch.Tensor:
    """
    Extract per-cell RNA-seq coverage over a genomic window at 1 bp resolution.

    Args:
        rna_data     : SnapATAC2 AnnData (scooby fork)
        cell_indices : list of integer indices into rna_data.obs
        chromosome   : e.g. 'chr19'
        start / end  : 0-based half-open genomic coordinates
        clip_soft    : soft-clip threshold (Borzoi / scooby default = 5.0)

    Returns:
        Tensor (n_cells, window_len) in squashed scale
    """
    window_len = end - start
    coverage   = np.zeros((len(cell_indices), window_len), dtype=np.float32)

    for i_out, i_cell in enumerate(cell_indices):
        # SnapATAC2 scooby fork: fetch_reads returns (read_start, read_length, strand)
        # Exact signature may differ slightly — verify against installed version
        try:
            cell_reads = snap.pp.fetch_reads(
                rna_data,
                cell_index=i_cell,
                chromosome=chromosome,
                start=start,
                end=end,
            )
        except AttributeError:
            # Fallback: try region query if fetch_reads API differs
            cell_reads = snap.pp.fetch_fragments(
                rna_data,
                cell_index=i_cell,
                region=f'{chromosome}:{start}-{end}',
            )

        for read_start, read_length, strand in cell_reads:
            rel_start = max(0, read_start - start)
            rel_end   = min(window_len, read_start + abs(read_length) - start)
            if rel_end > rel_start:
                coverage[i_out, rel_start:rel_end] += 1.0

    # Squashed-scale normalisation (same as Borzoi / scooby)
    squashed = np.where(
        coverage <= clip_soft,
        coverage,
        clip_soft + np.log(coverage - clip_soft + 1.0)
    )
    return torch.tensor(squashed, dtype=torch.float32)


def reverse_squash(x: torch.Tensor, clip_soft: float = 5.0) -> torch.Tensor:
    """Inverse squash for converting predictions back to raw read counts."""
    return torch.where(
        x <= clip_soft,
        x,
        torch.exp(x - clip_soft) + clip_soft - 1.0
    )


print('Coverage utilities defined.')

## 5 — AlphaGenome Sequence Embedding (Cached)

In [ ]:
def get_alphagenome_sequence_embedding(
    chromosome: str,
    interval_start: int,
    interval_end: int,
    dna_model,
) -> torch.Tensor:
    """
    Query AlphaGenome API for all RNA-seq tracks over a 1 Mb window.

    NOTE (API prototype): we retrieve bulk RNA-seq predictions (667 tracks)
    and use them as a fixed 'sequence embedding'. The cell-specific decoder
    learns to re-weight these bulk tracks per cell type.

    With local weights: replace with internal backbone embeddings (1920 dims)
    by updating n_seq_features=1920 everywhere.

    Returns:
        Tensor (sequence_length, n_rna_tracks)
    """
    interval = genome.Interval(
        chromosome=chromosome,
        start=interval_start,
        end=interval_end,
    )
    output = dna_model.predict_interval(
        interval=interval,
        requested_outputs=[dna_client.OutputType.RNA_SEQ],
    )
    # output.rna_seq.values: numpy (1048576, n_tracks)
    return torch.tensor(output.rna_seq.values, dtype=torch.float32)


def get_or_cache_embedding(
    chromosome: str,
    interval_start: int,
    interval_end: int,
    dna_model,
    cache_dir: str,
) -> torch.Tensor:
    """Load from Drive cache if available; otherwise query API and save."""
    key        = f'{chromosome}_{interval_start}_{interval_end}'
    cache_file = Path(cache_dir) / f'{key}.pt'

    if cache_file.exists():
        return torch.load(cache_file, map_location='cpu')

    emb = get_alphagenome_sequence_embedding(
        chromosome, interval_start, interval_end, dna_model
    )
    torch.save(emb, cache_file)
    return emb


print('Sequence embedding functions defined.')
print(f'Cache directory: {CACHE_DIR}')

## 6 — Model Architecture

In [ ]:
class HypernetworkMLP(nn.Module):
    """
    Generates per-cell convolutional filter weights from a cell embedding.

    Architecture (matches scooby):
        14 → 64 → 128 → 256 → (n_seq_features + 1) × num_output_tracks

    n_seq_features = 667  for API prototype  (AlphaGenome RNA-seq tracks)
    n_seq_features = 1920 for local weights  (AlphaGenome internal embedding)
    """

    def __init__(
        self,
        embedding_dim:     int   = 14,
        n_seq_features:    int   = 667,
        num_output_tracks: int   = 1,
        dropout:           float = 0.2,
    ):
        super().__init__()
        self.n_seq_features    = n_seq_features
        self.num_output_tracks = num_output_tracks
        filter_output_size = (n_seq_features + 1) * num_output_tracks

        self.mlp = nn.Sequential(
            nn.Linear(embedding_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, filter_output_size),
        )

    def forward(
        self,
        cell_embedding: torch.Tensor,   # (batch, embedding_dim) or (embedding_dim,)
    ) -> tuple:
        """
        Returns:
            weights (batch, n_seq_features, num_output_tracks)
            biases  (batch, 1, num_output_tracks)
        """
        if cell_embedding.dim() == 1:
            cell_embedding = cell_embedding.unsqueeze(0)

        out = self.mlp(cell_embedding)          # (batch, filter_output_size)
        n_w = self.n_seq_features * self.num_output_tracks
        weights = out[:, :n_w].view(-1, self.n_seq_features, self.num_output_tracks)
        biases  = out[:, n_w:].view(-1, 1, self.num_output_tracks)
        return weights, biases

In [ ]:
class CellSpecificDecoder(nn.Module):
    """
    Dynamic position-wise convolution conditioned on cell embedding.

    For each position p:
        output[cell, p] = seq_emb[p] @ filter_weights[cell] + filter_bias[cell]

    Applied only to exon-overlapping positions for efficiency.
    """

    def __init__(
        self,
        n_seq_features:    int   = 667,
        embedding_dim:     int   = 14,
        num_output_tracks: int   = 1,
        dropout:           float = 0.2,
    ):
        super().__init__()
        self.hypernetwork = HypernetworkMLP(
            embedding_dim=embedding_dim,
            n_seq_features=n_seq_features,
            num_output_tracks=num_output_tracks,
            dropout=dropout,
        )

    def forward(
        self,
        sequence_embedding: torch.Tensor,  # (seq_len, n_seq_features)
        cell_embedding:     torch.Tensor,  # (n_cells, embedding_dim)
        exon_mask:          torch.Tensor = None,  # (seq_len,) bool
    ) -> torch.Tensor:
        """
        Returns:
            predictions (n_cells, seq_len) — predicted coverage
        """
        n_cells = cell_embedding.shape[0]
        seq_len = sequence_embedding.shape[0]
        weights, biases = self.hypernetwork(cell_embedding)
        # weights: (n_cells, n_seq_features, 1)
        # biases:  (n_cells, 1, 1)

        if exon_mask is not None:
            seq_exon   = sequence_embedding[exon_mask]          # (n_exon, n_seq_features)
            pred_exon  = torch.einsum('pf,cft->cpt', seq_exon, weights) + biases
            pred_exon  = pred_exon.squeeze(-1)                   # (n_cells, n_exon)
            predictions = torch.zeros(n_cells, seq_len, device=cell_embedding.device)
            predictions[:, exon_mask] = pred_exon
        else:
            predictions = torch.einsum('pf,cft->cpt', sequence_embedding, weights) + biases
            predictions = predictions.squeeze(-1)                # (n_cells, seq_len)

        return predictions

In [ ]:
class AlphaGenomeScRNAModel(nn.Module):
    """
    Full model: AlphaGenome backbone (fixed/API) + cell-specific decoder.

    API prototype  : only CellSpecificDecoder is trained.
    Local weights  : add LoRA adapters to backbone and train jointly.
    """

    def __init__(
        self,
        n_seq_features: int   = 667,
        embedding_dim:  int   = 14,
        dropout:        float = 0.2,
    ):
        super().__init__()
        self.decoder = CellSpecificDecoder(
            n_seq_features=n_seq_features,
            embedding_dim=embedding_dim,
            num_output_tracks=1,
            dropout=dropout,
        )

    def forward(
        self,
        sequence_embedding: torch.Tensor,
        cell_embeddings:    torch.Tensor,
        exon_mask:          torch.Tensor = None,
    ) -> torch.Tensor:
        return self.decoder(sequence_embedding, cell_embeddings, exon_mask)


def count_parameters(model: nn.Module):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Total parameters    : {total:,}')
    print(f'Trainable parameters: {trainable:,}')
    return trainable


# ── Hyperparameters ──────────────────────────────────────────────────────────
N_SEQ_FEATURES = 667   # API prototype; set to 1920 with local weights
EMBEDDING_DIM  = 14    # scVI latent dimension
DROPOUT        = 0.2

model = AlphaGenomeScRNAModel(
    n_seq_features=N_SEQ_FEATURES,
    embedding_dim=EMBEDDING_DIM,
    dropout=DROPOUT,
).to(device)

count_parameters(model)

## 7 — Loss Function

In [ ]:
def poisson_multinomial_loss(
    pred:         torch.Tensor,   # (n_cells, seq_len) raw scale
    target:       torch.Tensor,   # (n_cells, seq_len) raw scale
    total_weight: float = 1.0,
    eps:          float = 1e-7,
) -> torch.Tensor:
    """
    Combined Poisson + multinomial loss (Borzoi / scooby formulation).

    Poisson term    : penalises errors in total read count per cell
    Multinomial term: penalises errors in coverage profile shape
    """
    pred   = pred   + eps
    target = target + eps

    pred_sum   = pred.sum(dim=-1, keepdim=True)
    target_sum = target.sum(dim=-1, keepdim=True)
    poisson_loss = (pred_sum - target_sum * torch.log(pred_sum)).mean()

    pred_norm   = pred   / pred_sum
    target_norm = target / target_sum
    multinomial_loss = -(target_norm * torch.log(pred_norm)).sum(dim=-1).mean()

    return total_weight * poisson_loss + multinomial_loss


print('Loss function defined.')

## 8 — PyTorch Dataset & DataLoaders

In [ ]:
class ScRNASeqCoverageDataset(torch.utils.data.Dataset):
    """
    Each item = one gene:
      - sequence embedding from AlphaGenome (cached)
      - n_cells_per_gene randomly sampled cells
      - per-cell 1 bp coverage profiles extracted from SnapATAC2 AnnData
      - exon mask for efficient decoding
    """

    def __init__(
        self,
        gene_df:           pd.DataFrame,
        rna_data,
        cell_embeddings:   torch.Tensor,
        cell_barcodes:     list,
        dna_model,
        exon_df:           pd.DataFrame = None,
        n_cells_per_gene:  int   = 64,
        split:             str   = 'train',
        cache_dir:         str   = '',
    ):
        self.gene_df         = gene_df[gene_df['split'] == split].reset_index(drop=True)
        self.rna_data        = rna_data
        self.cell_embeddings = cell_embeddings
        self.n_cells         = len(cell_barcodes)
        self.dna_model       = dna_model
        self.exon_df         = exon_df
        self.n_cells_per_gene = n_cells_per_gene
        self.cache_dir       = Path(cache_dir)
        self.cache_dir.mkdir(parents=True, exist_ok=True)

    def __len__(self):
        return len(self.gene_df)

    def _get_embedding(self, row):
        return get_or_cache_embedding(
            chromosome=row['chromosome'],
            interval_start=int(row['interval_start']),
            interval_end=int(row['interval_end']),
            dna_model=self.dna_model,
            cache_dir=str(self.cache_dir),
        )

    def _get_exon_mask(self, row) -> torch.Tensor:
        iv_start   = int(row['interval_start'])
        iv_end     = int(row['interval_end'])
        window_len = iv_end - iv_start
        mask       = torch.zeros(window_len, dtype=torch.bool)

        if self.exon_df is None:
            g_start = max(0, int(row['gene_start']) - iv_start)
            g_end   = min(window_len, int(row['gene_end']) - iv_start)
            mask[g_start:g_end] = True
            return mask

        gene_exons = self.exon_df[self.exon_df['gene_id'] == row['gene_id']]
        for _, exon in gene_exons.iterrows():
            rs = max(0, int(exon['exon_start']) - iv_start)
            re = min(window_len, int(exon['exon_end']) - iv_start)
            if re > rs:
                mask[rs:re] = True
        return mask

    def __getitem__(self, idx):
        row          = self.gene_df.iloc[idx]
        seq_emb      = self._get_embedding(row)
        cell_indices = np.random.choice(self.n_cells, size=self.n_cells_per_gene, replace=False)
        cell_emb     = self.cell_embeddings[cell_indices]

        coverage = extract_coverage_1bp(
            rna_data=self.rna_data,
            cell_indices=cell_indices.tolist(),
            chromosome=row['chromosome'],
            start=int(row['interval_start']),
            end=int(row['interval_end']),
        )
        exon_mask = self._get_exon_mask(row)

        return {
            'seq_emb':   seq_emb,
            'cell_emb':  cell_emb,
            'coverage':  coverage,
            'exon_mask': exon_mask,
            'gene_name': row['gene_name'],
            'chromosome': row['chromosome'],
        }

In [ ]:
N_CELLS_PER_GENE = 64   # cells sampled per gene per training step

dataset_kwargs = dict(
    gene_df=gene_df,
    rna_data=rna_data,
    cell_embeddings=cell_embeddings,
    cell_barcodes=cell_barcodes,
    dna_model=dna_model,
    exon_df=exon_df,
    n_cells_per_gene=N_CELLS_PER_GENE,
    cache_dir=CACHE_DIR,
)

train_dataset = ScRNASeqCoverageDataset(**dataset_kwargs, split='train')
val_dataset   = ScRNASeqCoverageDataset(**dataset_kwargs, split='val')
test_dataset  = ScRNASeqCoverageDataset(**dataset_kwargs, split='test')

# batch_size=1: each item is already a (gene × 64 cells) block;
# sequences are too large to stack across genes
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=1, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = torch.utils.data.DataLoader(val_dataset,   batch_size=1, shuffle=False, num_workers=2)
test_loader  = torch.utils.data.DataLoader(test_dataset,  batch_size=1, shuffle=False, num_workers=2)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

## 9 — Baseline Evaluation (AlphaGenome Bulk, No Cell Decoder)

In [ ]:
def evaluate_baseline(
    gene_df_split: pd.DataFrame,
    rna_data,
    cell_barcodes: list,
    cell_types:    list,
    dna_model,
    n_genes: int = 50,
) -> pd.DataFrame:
    """Bulk AlphaGenome prediction (mean across RNA-seq tracks) vs pseudobulk per cell type."""
    results          = []
    cell_type_array  = np.array(cell_types)
    unique_cts       = np.unique(cell_type_array)

    for _, row in tqdm(gene_df_split.head(n_genes).iterrows(), total=n_genes, desc='Baseline'):
        chrom, iv_start, iv_end = row['chromosome'], int(row['interval_start']), int(row['interval_end'])

        interval = genome.Interval(chrom, iv_start, iv_end)
        try:
            output = dna_model.predict_interval(
                interval=interval,
                requested_outputs=[dna_client.OutputType.RNA_SEQ],
            )
        except Exception as e:
            print(f'API error for {row["gene_name"]}: {e}')
            continue

        g_start = max(0, int(row['gene_start']) - iv_start)
        g_end   = min(iv_end - iv_start, int(row['gene_end']) - iv_start)
        bulk_pred = float(output.rna_seq.values[g_start:g_end].sum(axis=0).mean())

        for ct in unique_cts:
            ct_indices = np.where(cell_type_array == ct)[0]
            if len(ct_indices) < 5:
                continue
            obs_cov   = extract_coverage_1bp(rna_data, ct_indices[:20].tolist(), chrom, iv_start, iv_end)
            obs_count = float(obs_cov[:, g_start:g_end].sum(dim=1).mean())
            results.append({'gene_name': row['gene_name'], 'cell_type': ct,
                            'baseline_pred': bulk_pred, 'observed_count': obs_count})

    return pd.DataFrame(results)


print('Running baseline evaluation on 50 val genes ...')
baseline_results = evaluate_baseline(
    gene_df_split=val_genes,
    rna_data=rna_data,
    cell_barcodes=cell_barcodes,
    cell_types=cell_types,
    dna_model=dna_model,
    n_genes=50,
)

r_baseline, _ = pearsonr(
    np.log1p(baseline_results['baseline_pred']),
    np.log1p(baseline_results['observed_count'])
)
print(f'Baseline Pearson R (log scale): {r_baseline:.4f}')
baseline_results.to_csv(f'{SAVE_DIR}/baseline_results.csv', index=False)

## 10 — Optimizer, Scheduler & Checkpoint Utilities

In [ ]:
N_EPOCHS   = 40
WARMUP_STEPS  = 1000
TOTAL_STEPS   = N_EPOCHS * len(train_loader)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=4e-4,           # scooby decoder learning rate
    weight_decay=1e-6,
)

def lr_lambda(step: int) -> float:
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    decay = (TOTAL_STEPS - step) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return max(0.0, decay)

scheduler    = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
global_step  = 0

print(f'Optimizer  : AdamW  lr=4e-4  wd=1e-6')
print(f'Schedule   : linear warmup ({WARMUP_STEPS} steps) + linear decay')
print(f'Total steps: {TOTAL_STEPS}')

In [ ]:
# ════════════════════════════════════════════════════════════════════
# STANDALONE CELL — run this independently after reconnection
# to reload model + optimizer + scheduler from the latest checkpoint
# ════════════════════════════════════════════════════════════════════

def save_checkpoint(
    model, optimizer, scheduler, epoch, global_step, val_r, save_dir, is_best=False
):
    ckpt = {
        'epoch': epoch, 'global_step': global_step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'val_pearson_r': val_r,
    }
    path = os.path.join(save_dir, f'checkpoint_epoch{epoch:03d}.pt')
    torch.save(ckpt, path)
    if is_best:
        torch.save(ckpt, os.path.join(save_dir, 'best_model.pt'))
        print(f'  ★ New best saved: epoch={epoch}  val_R={val_r:.4f}')


def load_checkpoint(model, optimizer, scheduler, checkpoint_path, device):
    """Restore training state after Colab disconnection."""
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    print(f'Resumed: epoch={ckpt["epoch"]}  step={ckpt["global_step"]}  val_R={ckpt["val_pearson_r"]:.4f}')
    return ckpt['epoch'], ckpt['global_step'], ckpt['val_pearson_r']


# ── RELOAD (uncomment after disconnection) ───────────────────────────────────
# CHECKPOINT_PATH = f'{SAVE_DIR}/best_model.pt'
# start_epoch, global_step, best_val_r = load_checkpoint(
#     model, optimizer, scheduler, CHECKPOINT_PATH, device
# )
# ─────────────────────────────────────────────────────────────────────────────

print('Checkpoint utilities defined.')

## 11 — Training Loop

In [ ]:
def evaluate_model(model, loader, device, n_batches=50) -> float:
    """Quick evaluation: mean across-gene Pearson R (log2 scale)."""
    model.eval()
    all_preds, all_targets = [], []

    with torch.no_grad():
        for i, batch in enumerate(loader):
            if i >= n_batches:
                break
            seq_emb  = batch['seq_emb'].squeeze(0).to(device)
            cell_emb = batch['cell_emb'].squeeze(0).to(device)
            coverage = batch['coverage'].squeeze(0).to(device)
            exon_mask = batch['exon_mask'].squeeze(0).to(device)

            pred = model(seq_emb, cell_emb, exon_mask)

            pred_counts   = reverse_squash(pred[:, exon_mask]).sum(dim=-1).cpu().numpy()
            target_counts = reverse_squash(coverage[:, exon_mask]).sum(dim=-1).cpu().numpy()

            all_preds.extend(pred_counts.tolist())
            all_targets.extend(target_counts.tolist())

    all_preds, all_targets = np.array(all_preds), np.array(all_targets)
    r, _ = pearsonr(np.log2(all_preds + 1), np.log2(all_targets + 1))
    return float(r)


print('Evaluation function defined.')

In [ ]:
history = {'epoch': [], 'train_loss': [], 'val_pearson_r': [], 'lr': []}
best_val_r  = -np.inf
start_epoch = 0   # change when resuming from checkpoint

for epoch in range(start_epoch, N_EPOCHS):
    model.train()
    epoch_losses = []

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1:02d}/{N_EPOCHS}')
    for batch in pbar:
        seq_emb   = batch['seq_emb'].squeeze(0).to(device)
        cell_emb  = batch['cell_emb'].squeeze(0).to(device)
        coverage  = batch['coverage'].squeeze(0).to(device)
        exon_mask = batch['exon_mask'].squeeze(0).to(device)

        optimizer.zero_grad()
        pred      = model(seq_emb, cell_emb, exon_mask)

        # Loss on exon positions only (in raw scale)
        pred_raw   = reverse_squash(pred[:, exon_mask])
        target_raw = reverse_squash(coverage[:, exon_mask])
        loss = poisson_multinomial_loss(pred_raw, target_raw, total_weight=1.0)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        global_step += 1

        epoch_losses.append(loss.item())
        pbar.set_postfix({'loss': f'{loss.item():.4f}',
                          'lr':   f'{scheduler.get_last_lr()[0]:.2e}'})

    val_r     = evaluate_model(model, val_loader, device, n_batches=100)
    mean_loss = float(np.mean(epoch_losses))
    is_best   = val_r > best_val_r
    if is_best:
        best_val_r = val_r

    history['epoch'].append(epoch + 1)
    history['train_loss'].append(mean_loss)
    history['val_pearson_r'].append(val_r)
    history['lr'].append(scheduler.get_last_lr()[0])

    print(f'Epoch {epoch+1:02d}: loss={mean_loss:.4f}  val_R={val_r:.4f}{"  ★" if is_best else ""}')

    save_checkpoint(
        model=model, optimizer=optimizer, scheduler=scheduler,
        epoch=epoch+1, global_step=global_step, val_r=val_r,
        save_dir=SAVE_DIR, is_best=is_best,
    )

history_df = pd.DataFrame(history)
history_df.to_csv(f'{SAVE_DIR}/training_history.csv', index=False)
print('\nTraining complete.')

## 12 — Test Set Evaluation (scooby metrics)

In [ ]:
# Load best checkpoint for final evaluation
best_ckpt = torch.load(f'{SAVE_DIR}/best_model.pt', map_location=device)
model.load_state_dict(best_ckpt['model_state_dict'])
model.eval()
print(f'Loaded best model: epoch={best_ckpt["epoch"]}  val_R={best_ckpt["val_pearson_r"]:.4f}')

In [ ]:
def evaluate_test_set(
    model, test_loader, rna_data, cell_embeddings, cell_types, device
) -> dict:
    """
    Full scooby-style test evaluation:
    1. Across-gene Pearson R per cell type (log2 scale)
    2. Between-cell-type deviation R (after removing gene and cell means)
    """
    model.eval()
    ct_array   = np.array(cell_types)
    unique_cts = np.unique(ct_array)
    results    = {ct: {'pred': [], 'obs': [], 'gene': []} for ct in unique_cts}

    with torch.no_grad():
        for batch in tqdm(test_loader, desc='Test evaluation'):
            seq_emb   = batch['seq_emb'].squeeze(0).to(device)
            exon_mask = batch['exon_mask'].squeeze(0).to(device)
            gene_name = batch['gene_name'][0]

            for ct in unique_cts:
                ct_idx = np.where(ct_array == ct)[0]
                if len(ct_idx) < 5:
                    continue
                sampled = ct_idx[:50]
                ct_emb  = cell_embeddings[sampled].to(device)

                obs_cov = extract_coverage_1bp(
                    rna_data=rna_data,
                    cell_indices=sampled.tolist(),
                    chromosome=batch['chromosome'][0],
                    start=int(seq_emb.shape[0] // 2),
                    end=int(seq_emb.shape[0] // 2 + exon_mask.sum().item()),
                )

                pred = model(seq_emb, ct_emb, exon_mask)
                results[ct]['pred'].append(reverse_squash(pred[:, exon_mask]).sum(dim=-1).mean().item())
                results[ct]['obs'].append(reverse_squash(obs_cov).sum(dim=-1).mean().item())
                results[ct]['gene'].append(gene_name)

    # Metric 1: across-gene Pearson R per cell type
    across_gene_r = {}
    for ct in unique_cts:
        if len(results[ct]['pred']) < 10:
            continue
        p = np.log2(np.array(results[ct]['pred']) + 1)
        o = np.log2(np.array(results[ct]['obs'])  + 1)
        across_gene_r[ct], _ = pearsonr(p, o)

    # Metric 2: between-cell-type deviation (after removing gene and cell means)
    all_cts   = [ct  for ct in unique_cts if ct in across_gene_r]
    all_genes = sorted(set(g for ct in all_cts for g in results[ct]['gene']))
    pred_mat  = np.full((len(all_cts), len(all_genes)), np.nan)
    obs_mat   = np.full((len(all_cts), len(all_genes)), np.nan)
    gene_idx  = {g: i for i, g in enumerate(all_genes)}

    for i, ct in enumerate(all_cts):
        for g, p, o in zip(results[ct]['gene'], results[ct]['pred'], results[ct]['obs']):
            j = gene_idx[g]
            pred_mat[i, j] = np.log2(p + 1)
            obs_mat[i, j]  = np.log2(o + 1)

    pred_dev = pred_mat - np.nanmean(pred_mat, axis=0, keepdims=True) \
                        - np.nanmean(pred_mat, axis=1, keepdims=True)
    obs_dev  = obs_mat  - np.nanmean(obs_mat,  axis=0, keepdims=True) \
                        - np.nanmean(obs_mat,  axis=1, keepdims=True)
    valid    = ~(np.isnan(pred_dev) | np.isnan(obs_dev))
    dev_r, _ = pearsonr(pred_dev[valid].ravel(), obs_dev[valid].ravel())

    mean_r = float(np.mean(list(across_gene_r.values())))
    print(f'Mean across-gene Pearson R : {mean_r:.4f}')
    print(f'Between-cell-type deviation R: {dev_r:.4f}')

    return {
        'across_gene_pearson_r':   across_gene_r,
        'mean_across_gene_r':      mean_r,
        'deviation_pearson_r':     dev_r,
        'results_by_celltype':     results,
    }


test_results = evaluate_test_set(
    model=model, test_loader=test_loader,
    rna_data=rna_data, cell_embeddings=cell_embeddings,
    cell_types=cell_types, device=device,
)

## 13 — Visualise Results

In [ ]:
# Reload history if resuming from Drive
if 'history_df' not in dir():
    history_df = pd.read_csv(f'{SAVE_DIR}/training_history.csv')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_df['epoch'], history_df['train_loss'], color='steelblue')
axes[0].set(xlabel='Epoch', ylabel='Poisson-multinomial loss', title='Training Loss')
axes[0].grid(alpha=0.3)

axes[1].plot(history_df['epoch'], history_df['val_pearson_r'], color='coral')
axes[1].axhline(r_baseline, ls='--', color='grey', label=f'Baseline R={r_baseline:.3f}')
axes[1].set(xlabel='Epoch', ylabel='Pearson R (log2)', title='Validation Pearson R')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Per-cell-type Pearson R: baseline vs finetuned
ct_names     = sorted(test_results['across_gene_pearson_r'].keys())
finetuned_r  = [test_results['across_gene_pearson_r'][ct] for ct in ct_names]

# Baseline per cell type (from baseline_results CSV)
baseline_df  = pd.read_csv(f'{SAVE_DIR}/baseline_results.csv')
baseline_r_ct = {}
for ct in ct_names:
    sub = baseline_df[baseline_df['cell_type'] == ct]
    if len(sub) > 5:
        r, _ = pearsonr(np.log1p(sub['baseline_pred']), np.log1p(sub['observed_count']))
        baseline_r_ct[ct] = r
    else:
        baseline_r_ct[ct] = np.nan

x, width = np.arange(len(ct_names)), 0.35
fig, ax  = plt.subplots(figsize=(max(10, len(ct_names)*0.8), 5))
ax.bar(x - width/2, [baseline_r_ct.get(ct, np.nan) for ct in ct_names],
       width, label='Baseline (bulk)', alpha=0.75, color='steelblue')
ax.bar(x + width/2, finetuned_r,
       width, label='Finetuned',       alpha=0.75, color='coral')
ax.set(xlabel='Cell type', ylabel='Pearson R (log2)',
       title='Across-gene Pearson R: Baseline vs Finetuned',
       ylim=(0, 1))
ax.set_xticks(x)
ax.set_xticklabels(ct_names, rotation=45, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/baseline_vs_finetuned_pearson_r.png', dpi=150, bbox_inches='tight')
plt.show()

## 14 — Save All Results & Model Weights

In [ ]:
# Final decoder weights (inference-only, no optimizer state)
torch.save(model.state_dict(), f'{SAVE_DIR}/finetuned_decoder_weights.pt')

evaluation_summary = {
    'best_val_pearson_r':            float(best_val_r),
    'test_mean_across_gene_pearson_r': float(test_results['mean_across_gene_r']),
    'test_deviation_pearson_r':       float(test_results['deviation_pearson_r']),
    'test_pearson_r_per_cell_type':   {k: float(v) for k, v in test_results['across_gene_pearson_r'].items()},
    'baseline_pearson_r':             float(r_baseline),
    'n_train_genes':  int(len(train_genes)),
    'n_val_genes':    int(len(val_genes)),
    'n_test_genes':   int(len(test_genes)),
    'n_cells':        int(len(cell_barcodes)),
    'embedding_dim':  EMBEDDING_DIM,
    'n_seq_features': N_SEQ_FEATURES,
    'n_epochs':       N_EPOCHS,
}

with open(f'{SAVE_DIR}/evaluation_summary.json', 'w') as fh:
    json.dump(evaluation_summary, fh, indent=2)

print(json.dumps(evaluation_summary, indent=2))
print(f'\n✅ All results saved to: {SAVE_DIR}')
for fname in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, fname))
    print(f'  {fname}  ({size/1024:.1f} KB)')

## 15 — Reload Weights (Standalone Cell)

Run this cell **alone** after a Colab disconnection to restore the model for inference without re-running the full notebook.

In [ ]:
# ════════════════════════════════════════════════════════════════════
# SELF-CONTAINED RELOAD CELL
# Run this cell independently after disconnection (no prior cells needed)
# ════════════════════════════════════════════════════════════════════
import torch, json, os
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/alphagenome_finetune'
SAVE_DIR   = f'{DRIVE_ROOT}/checkpoints'
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Re-instantiate model (must match training config exactly) ─────────────
# Copy HypernetworkMLP, CellSpecificDecoder, AlphaGenomeScRNAModel
# definitions here if running truly standalone, or import from a saved module.
# For brevity, assume classes are already defined in this session.

model = AlphaGenomeScRNAModel(
    n_seq_features=667,
    embedding_dim=14,
    dropout=0.2,
).to(device)

# Option A: load best checkpoint (full training state)
ckpt = torch.load(f'{SAVE_DIR}/best_model.pt', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f"Best model loaded: epoch={ckpt['epoch']}  val_R={ckpt['val_pearson_r']:.4f}")

# Option B: inference-only weights (lighter)
# model.load_state_dict(torch.load(f'{SAVE_DIR}/finetuned_decoder_weights.pt', map_location=device))

model.eval()
print('Model ready for inference.')

with open(f'{SAVE_DIR}/evaluation_summary.json') as fh:
    summary = json.load(fh)
print(json.dumps(summary, indent=2))